**Select chat model**

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

# Load the .env file
load_dotenv()
# assign key from env to langchain/openai


from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

print("API_KEY: ", os.getenv("OPENAI_API_KEY"))

# Make sure your API key is set in the environment
api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize model
llm = ChatOpenAI(
    model="gpt-4.1-nano",  # Make sure this model name is valid in your OpenAI account
    base_url="https://openrouter.ai/api/v1",
    temperature=0.6,
    api_key=api_key
)

API_KEY:  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


**OpenAI llm**

In [ ]:

# Create message list
messages = [
    SystemMessage(content="Say Hello in German"),
    HumanMessage(content="Can you speak German?")
]

# Call the model
response = llm.invoke(messages)

# Output result
print(response.content)

API_KEY:  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
Hallo! Ja, ich kann auf Deutsch sprechen.


**MistralAI llm**

In [14]:
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

model = init_chat_model("mistral-small", model_provider="mistralai", temperature=0.2)

from langchain_mistralai import ChatMistralAI
llm = ChatMistralAI(model="mistral-small")

response = model.invoke("Say hello in German")
response.usage_metadata
print(response)

content='The word "hello" can be translated to "hallo" in German. So, if you want to say hello in German, you can simply say "hallo"! This is a versatile and casual greeting that can be used in many different situations.\n\nHowever, it\'s worth noting that there are many other ways to say hello in German, depending on the time of day, the level of formality, and the relationship between the speakers. For example, "guten Tag" is a more formal way to say hello, and is often used in business settings or when meeting someone for the first time. "Guten Morgen" means "good morning," while "guten Abend" means "good evening."\n\nSo while "hallo" is a great way to say hello in many casual situations, it\'s always good to have a few other greetings up your sleeve as well!' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 13, 'total_tokens': 209, 'completion_tokens': 196}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'stop'} id='run--4f965625-b

**invoke structured prompt**
- generate n idioms in different german level

In [ ]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative german Teacher. You generate german idioms accordings to given topic for given level"),
    ("human", "Provide {number_idioms} idioms with {topic} for level A2"),
])
messages = prompt.format_messages(number_idioms=5, topic="food")

response = model.invoke(messages)
print(response.content)

### Generate idioms with given user's input, and put up a question

In [6]:
# Using PromptTemplate to set up prompt
from langchain.prompts import PromptTemplate
# call chain functions
from langchain.chains import LLMChain
# define model -> is done above

# define and chain idiom template
idiom_prompt = PromptTemplate(
    input_variables=["nbr_idioms", "topic", "level"],
    template =(
    "You are a creative German teacher. You generate German idioms according to a given topic for a given level.\n\n"
    "Provide {nbr_idioms} idioms about {topic} for level {level}."
    "After providing reponse ask user to make an example with one of given idiom"
    )
)
idiom_chain = LLMChain(llm=llm, prompt=idiom_prompt)

# fetch user's input
nbr_idioms = input("Enter number: ")
topic = input("Enter topic: ")
level = input("Enter level: ")

# collect user input
user_input = {
    "nbr_idioms": nbr_idioms,
    "topic": topic,
    "level": level
}

# run and print the chain
idiom_response = idiom_chain.invoke(user_input)
# print("\nGenerated idioms: \n", idiom_response)



Generated idioms: 
 {'nbr_idioms': '2', 'topic': 'weathe', 'level': 'a1', 'text': 'Natürlich! Hier sind zwei deutsche Redewendungen zum Thema Wetter für das Niveau A1:\n\n1. **Es ist ein Sonnentag.**  \n   (Es ist schönes Wetter, die Sonne scheint.)\n\n2. **Es regnet Katzen und Hunde.**  \n   (Es regnet sehr stark.)\n\nMöchtest du ein Beispiel mit einer dieser Redewendungen machen?'}


### generate evaluation based on the answer given by user

In [9]:
# from langchain.prompts import PromptTemplate
# define and chain evaluation_prompt
evaluation_prompt = PromptTemplate(
    input_variable = ["user_example", "idiom"],
    template=(
        "You are a German language teacher. Evaluate following sentece:\n"
        "\"{user_example}\"\n"
        "Did the user correctly use the idiom \"{idiom}\"? Provide detailed feedback in simple language" 
    )
) 
evaluation_chain = LLMChain(llm=llm, prompt = evaluation_prompt)

#  Ask user to pick an idiom and write a sentence
idiom_chosen = input("\nChoose one idiom from above to use: ")
user_example = input(f"Write a sentence using the idiom '{idiom_chosen}': ")

# Evaluate the sentence
evaluation_response = evaluation_chain.run({
    "user_example": user_example,
    "idiom": idiom_chosen
})
print("\nEvaluation:\n", evaluation_response)



Evaluation:
 Der Satz ist fast richtig, aber es gibt einen kleinen Fehler. Das Wort "sonigen" ist falsch geschrieben. Es sollte "sonnigen" heißen, weil es sich um die Form von "sonnig" handelt, also "nach einem sonnigen Tag".

Außerdem hast du die Redewendung "Es regnet Katzen und Hunde" richtig benutzt. Diese bedeutet, dass es sehr stark regnet. 

Korrekt wäre also:
„Nach einem sonnigen Tag regnet es Katzen und Hunde.“

Super gemacht! Nur die Schreibweise von „sonnigen“ muss noch korrigiert werden.


### Training

In [11]:
# Step 2: prepare a NEW prompt template
matching_prompt = PromptTemplate(
    input_variables=["idioms"],
    template=(
        "Create a matching exercise for these German idioms and their context meaning in English.\n"
        "Idioms:\n{idioms}\n\n"
        "Return two numbered lists: one with German idioms, one with English meanings in random order."
    )
)

# Step 3: create a NEW chain for the matching exercise
matching_chain = LLMChain(llm=llm, prompt=matching_prompt)
# Step 4: format idioms for prompt
german_idioms = [idiom.split(" - ")[0].strip() for idiom in idioms]
idioms_text = "\n".join(f"{i+1}. {idiom}" for i, idiom in enumerate(german_idioms))
prompt_input = {"idioms": idioms_text}

# Step 5: run the new chain
exercise_output = matching_chain.invoke(prompt_input)

print(exercise_output)

{'idioms': '1. nbr_idioms\n2. topic\n3. level\n4. text', 'text': "Sure! Here's a matching exercise with the German idioms and their English meanings in random order.\n\n**German Idioms:**\n1. *Die Katze im Sack kaufen*  \n2. *Jemandem die Daumen drücken*  \n3. *Das ist mir Wurst*  \n4. *Aus den Augen, aus dem Sinn*  \n\n**English Meanings:**\nA. To wish someone good luck or hope for their success.  \nB. To buy something without inspecting it first, risking a bad surprise.  \nC. To be indifferent or not care about something.  \nD. To forget about someone or something after losing sight of them."}


In [8]:
matching_evaluation_prompt = PromptTemplate(
    input_variable = ["user_awser", "exercise"],
    template=(
        "You are a German language teacher. Evaluate following answer:\n"
        "\"{user_answer}\"\n"
        "Did the user correctly match german idioms to their explainations \"{exercise}\"?"
        "Provide concise feedback" 
    )
) 
evaluation_chain = LLMChain(llm=llm, prompt = matching_evaluation_prompt)

#  Ask user to pick an idiom and write a sentence
exercise = exercise_output
user_example = input(f"Write a sentence using the idiom '{exercise}': ")

# Evaluate the sentence
matching_evaluation_response = evaluation_chain.run({
    "user_example": user_example,
    "idiom": idiom_chosen
})
print("\nEvaluation:\n", matching_evaluation_response)

NameError: name 'exercise_output' is not defined